# MOSAIC Kaggle Validation Suite

This notebook runs the language-model, operator, prefix-reuse, workload, and Triton experiments used in the MOSAIC evaluation. It exports CSV, JSON, figures, LaTeX tables, checkpoints, and a result archive.

Kaggle setting: GPU enabled, Internet enabled.

In [ ]:
import os, sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.48,<5", "datasets>=3.2,<5", "accelerate>=1.2,<2",
    "tokenizers>=0.20,<1", "sentencepiece>=0.2,<1",
    "safetensors>=0.4,<1", "scipy>=1.11,<2", "psutil>=5.9,<7"
])

In [ ]:
import gc, hashlib, json, math, platform, random, time, warnings, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib import pyplot as plt
from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainers
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/mosaic_validation")
RESULTS, FIGURES, CHECKPOINTS, CACHE = [ROOT / x for x in ["results", "figures", "checkpoints", "cache"]]
for p in [ROOT, RESULTS, FIGURES, CHECKPOINTS, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
RUN_LEVEL = "smoke"  # smoke -> pipeline check, paper -> publication run
RUN_SUBWORD_TRAINING = True
RUN_PRETRAINED_BENCHMARKS = True
RUN_TRITON_BENCHMARKS = True
USER_TRACE_JSONL = ""

PROFILES = {
    "smoke": dict(
        seeds=[17], vocab_size=4096, context=128, batch_size=8, grad_accum=1,
        train_steps=100, eval_batches=20, max_train_tokens=500_000,
        max_valid_tokens=100_000, d_model=128, n_layers=2, lr=3e-4,
        warmup_steps=20, prefill_contexts=[256, 512], prefill_repeats=5
    ),
    "paper": dict(
        seeds=[17, 23, 41], vocab_size=8192, context=256, batch_size=12, grad_accum=2,
        train_steps=1800, eval_batches=160, max_train_tokens=8_000_000,
        max_valid_tokens=800_000, d_model=192, n_layers=4, lr=2e-4,
        warmup_steps=150, prefill_contexts=[256, 512, 1024, 2048],
        prefill_repeats=20
    ),
}
CFG = PROFILES[RUN_LEVEL]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def params(model):
    return sum(p.numel() for p in model.parameters())

manifest = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "run_level": RUN_LEVEL,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "config": CFG,
}
(RESULTS / "environment.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

## 1. WikiText-103 and byte-level BPE

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
train_texts, valid_texts = ds["train"]["text"], ds["validation"]["text"]

tok_path = CACHE / f"bpe_{CFG['vocab_size']}.json"
if tok_path.exists():
    tokenizer = Tokenizer.from_file(str(tok_path))
else:
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=CFG["vocab_size"], min_frequency=2,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
    )
    def iterator():
        seen = 0
        budget = 8_000_000 if RUN_LEVEL == "paper" else 800_000
        for text in train_texts:
            if text.strip():
                yield text
                seen += len(text)
                if seen >= budget: break
    tokenizer.train_from_iterator(iterator(), trainer=trainer)
    tokenizer.save(str(tok_path))

VOCAB = tokenizer.get_vocab_size()
BOS = tokenizer.token_to_id("<bos>")
EOS = tokenizer.token_to_id("<eos>")

def encode(texts, limit):
    ids = []
    for text in texts:
        if not text.strip(): continue
        piece = [BOS] + tokenizer.encode(text).ids + [EOS]
        ids.extend(piece[:max(0, limit - len(ids))])
        if len(ids) >= limit: break
    return torch.tensor(ids, dtype=torch.long)

train_ids = encode(train_texts, CFG["max_train_tokens"])
valid_ids = encode(valid_texts, CFG["max_valid_tokens"])
print("vocab", VOCAB, "train", train_ids.numel(), "valid", valid_ids.numel())

In [ ]:
class RandomBlocks(Dataset):
    def __init__(self, ids, context, length):
        self.ids, self.context, self.length = ids, context, length
    def __len__(self): return self.length
    def __getitem__(self, _):
        s = torch.randint(0, self.ids.numel() - self.context - 1, (1,)).item()
        z = self.ids[s:s+self.context+1]
        return z[:-1], z[1:]

class FixedBlocks(Dataset):
    def __init__(self, ids, context, max_blocks):
        self.context = context
        self.length = min((ids.numel()-1)//context, max_blocks)
        self.ids = ids[:self.length*context+1]
    def __len__(self): return self.length
    def __getitem__(self, i):
        s = i*self.context
        z = self.ids[s:s+self.context+1]
        return z[:-1], z[1:]

train_ds = RandomBlocks(train_ids, CFG["context"], CFG["train_steps"]*CFG["batch_size"]*CFG["grad_accum"])
valid_ds = FixedBlocks(valid_ids, CFG["context"], CFG["eval_batches"]*CFG["batch_size"])

## 2. MOSAIC and matched baselines

In [ ]:
def causal_dw(conv, x):
    y = conv(x.transpose(1,2))
    return y[:,:,:x.shape[1]].transpose(1,2)

def affine_scan(a, b):
    a32, b32 = a.float(), b.float()
    p = torch.exp(torch.cumsum(torch.log(a32.clamp_min(1e-6)), dim=1))
    return (p * torch.cumsum(b32 / p.clamp_min(1e-20), dim=1)).to(b.dtype)

class MOSAIC(nn.Module):
    def __init__(self, v, d):
        super().__init__()
        self.emb = nn.Embedding(v,d)
        self.dw = nn.Conv1d(d,d,5,groups=d,padding=4)
        self.pw = nn.Linear(d,d)
        self.ap = nn.Linear(d,d); self.bp = nn.Linear(d,d)
        self.mix = nn.Linear(2*d,d); self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d,v,bias=False); self.head.weight = self.emb.weight
    def forward(self, ids):
        e = self.emb(ids)
        a = .980 + .019*torch.sigmoid(self.ap(e))
        b = torch.tanh(self.bp(e))
        local = self.pw(torch.tanh(causal_dw(self.dw,e)))
        state = affine_scan(a,b)
        return self.head(self.norm(torch.tanh(self.mix(torch.cat([local,state],-1)))))

class GRU(nn.Module):
    def __init__(self,v,d):
        super().__init__()
        self.emb=nn.Embedding(v,d); self.rnn=nn.GRU(d,d,batch_first=True)
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,v,bias=False); self.head.weight=self.emb.weight
    def forward(self,ids):
        y,_=self.rnn(self.emb(ids)); return self.head(self.norm(y))

class TCN(nn.Module):
    def __init__(self,v,d,layers):
        super().__init__()
        self.emb=nn.Embedding(v,d)
        self.convs=nn.ModuleList([nn.Conv1d(d,d,3,dilation=2**i,padding=2*(2**i)) for i in range(layers)])
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,v,bias=False); self.head.weight=self.emb.weight
    def forward(self,ids):
        x=self.emb(ids).transpose(1,2); t=ids.shape[1]
        for c in self.convs: x=x+torch.tanh(c(x)[:,:,:t])
        return self.head(self.norm(x.transpose(1,2)))

class Transformer(nn.Module):
    def __init__(self,v,d,layers,context):
        super().__init__()
        heads=6 if d%6==0 else 4
        self.emb=nn.Embedding(v,d); self.pos=nn.Embedding(context,d)
        layer=nn.TransformerEncoderLayer(d,heads,4*d,dropout=0.0,activation="gelu",batch_first=True,norm_first=True)
        self.net=nn.TransformerEncoder(layer,layers)
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,v,bias=False); self.head.weight=self.emb.weight
    def forward(self,ids):
        t=ids.shape[1]; p=torch.arange(t,device=ids.device)
        x=self.emb(ids)+self.pos(p)[None]
        mask=torch.triu(torch.full((t,t),float("-inf"),device=ids.device),1)
        return self.head(self.norm(self.net(x,mask=mask)))

def build():
    d=CFG["d_model"]
    return {
        "MOSAIC": MOSAIC(VOCAB,d),
        "GRU": GRU(VOCAB,d),
        "TCN": TCN(VOCAB,d,CFG["n_layers"]),
        "Transformer": Transformer(VOCAB,d,CFG["n_layers"],CFG["context"]),
    }

pd.DataFrame([{"model":n,"parameters":params(m)} for n,m in build().items()])

## 3. Training, evaluation, and paired bootstrap

In [ ]:
def lr_at(step):
    if step < CFG["warmup_steps"]:
        return CFG["lr"]*(step+1)/CFG["warmup_steps"]
    q=(step-CFG["warmup_steps"])/max(1,CFG["train_steps"]-CFG["warmup_steps"])
    return CFG["lr"]*.5*(1+math.cos(math.pi*q))

def bootstrap(values, seed=1, draws=3000):
    rng=np.random.default_rng(seed); values=np.asarray(values)
    means=[rng.choice(values,len(values),replace=True).mean() for _ in range(draws)]
    return np.quantile(means,[.025,.975])

def train_one(name, seed):
    seed_all(seed); model=build()[name].to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],betas=(.9,.95),weight_decay=.1)
    scaler=torch.amp.GradScaler("cuda",enabled=torch.cuda.is_available() and AMP_DTYPE==torch.float16)
    loader=DataLoader(train_ds,batch_size=CFG["batch_size"],num_workers=2,drop_last=True)
    it=iter(loader); rows=[]; opt.zero_grad(set_to_none=True); start=time.perf_counter()
    for step in range(CFG["train_steps"]):
        for g in opt.param_groups: g["lr"]=lr_at(step)
        loss_sum=0
        for _ in range(CFG["grad_accum"]):
            x,y=next(it); x=x.to(DEVICE); y=y.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=torch.cuda.is_available()):
                logits=model(x); loss=F.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1))
            scaler.scale(loss/CFG["grad_accum"]).backward(); loss_sum+=float(loss.detach())
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step==0 or (step+1)%max(10,CFG["train_steps"]//20)==0:
            rows.append(dict(model=name,seed=seed,step=step+1,train_loss=loss_sum/CFG["grad_accum"],elapsed_s=time.perf_counter()-start))
            print(rows[-1])
    torch.save({"state_dict":model.state_dict(),"model":name,"seed":seed,"config":CFG},CHECKPOINTS/f"{name}_seed{seed}.pt")
    return model,rows

@torch.no_grad()
def evaluate(model):
    model.eval(); losses=[]; tokens=0; start=time.perf_counter()
    for x,y in DataLoader(valid_ds,batch_size=CFG["batch_size"],num_workers=2):
        x=x.to(DEVICE); y=y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=torch.cuda.is_available()):
            logits=model(x)
        z=F.cross_entropy(logits.float().transpose(1,2),y,reduction="none").mean(1)
        losses.extend(z.cpu().tolist()); tokens+=y.numel()
    sync(); arr=np.asarray(losses); ci=bootstrap(arr)
    return dict(loss=float(arr.mean()),loss_ci_low=float(ci[0]),loss_ci_high=float(ci[1]),
                bpc=float(arr.mean()/math.log(2)),perplexity=float(math.exp(arr.mean())),
                tokens_per_s=tokens/(time.perf_counter()-start),vector=arr)

In [ ]:
quality=[]; curves=[]; vectors={}
if RUN_SUBWORD_TRAINING:
    for seed in CFG["seeds"]:
        for name in ["MOSAIC","GRU","TCN","Transformer"]:
            model,rows=train_one(name,seed); curves+=rows; ev=evaluate(model)
            vectors[(name,seed)]=ev.pop("vector")
            quality.append({"model":name,"seed":seed,"parameters":params(model),**ev})
            del model; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

quality_df=pd.DataFrame(quality); curves_df=pd.DataFrame(curves)
quality_df.to_csv(RESULTS/"subword_quality.csv",index=False)
curves_df.to_csv(RESULTS/"training_curves.csv",index=False)

paired=[]
for seed in CFG["seeds"]:
    if ("MOSAIC",seed) not in vectors: continue
    for base in ["GRU","TCN","Transformer"]:
        d=vectors[("MOSAIC",seed)]-vectors[(base,seed)]; ci=bootstrap(d,seed+100)
        paired.append({"seed":seed,"comparison":f"MOSAIC - {base}","mean_loss_difference":float(d.mean()),"ci_low":float(ci[0]),"ci_high":float(ci[1])})
paired_df=pd.DataFrame(paired); paired_df.to_csv(RESULTS/"paired_quality.csv",index=False)
quality_df

In [ ]:
if not quality_df.empty:
    summary=quality_df.groupby("model").agg(
        loss_mean=("loss","mean"),loss_sd=("loss","std"),bpc_mean=("bpc","mean"),
        bpc_sd=("bpc","std"),perplexity_mean=("perplexity","mean"),
        tokens_per_s_mean=("tokens_per_s","mean"),parameters=("parameters","first")
    ).reset_index()
    summary.to_csv(RESULTS/"subword_quality_summary.csv",index=False)
    summary.plot(x="model",y="bpc_mean",kind="bar",legend=False,yerr="bpc_sd",capsize=4)
    plt.ylabel("Validation bits per token"); plt.tight_layout()
    plt.savefig(FIGURES/"subword_quality.png",dpi=180); plt.show()

## 4. Affine operator composition and powering

In [ ]:
def compose(op2,op1):
    a2,b2=op2; a1,b1=op1
    return a2*a1, a2*b1+b2

def compile_op(a,b):
    out=(torch.ones_like(a[0]),torch.zeros_like(b[0]))
    for i in range(a.shape[0]): out=compose((a[i],b[i]),out)
    return out

def power(op,n):
    out=(torch.ones_like(op[0]),torch.zeros_like(op[1])); base=op
    while n:
        if n&1: out=compose(base,out)
        base=compose(base,base); n>>=1
    return out

rows=[]
for dtype in [torch.float32,torch.bfloat16,torch.float16]:
    if dtype==torch.float16 and DEVICE.type=="cpu": continue
    for L in [16,64,256]:
        for r in [1,8,64,512]:
            seed_all(L+r); d=256
            a=(.995+.004*torch.rand(L,d,device=DEVICE)).to(dtype)
            b=(.02*torch.randn(L,d,device=DEVICE)).to(dtype)
            h0=torch.randn(d,device=DEVICE,dtype=dtype); direct=h0.clone()
            for _ in range(r):
                for t in range(L): direct=a[t]*direct+b[t]
            op=power(compile_op(a,b),r); cached=op[0]*h0+op[1]
            err=(direct.float()-cached.float()).abs()
            rows.append({"dtype":str(dtype).replace("torch.",""),"segment_length":L,"repetitions":r,
                         "context_tokens":L*r,"max_abs_error":float(err.max()),"mean_abs_error":float(err.mean())})
correctness_df=pd.DataFrame(rows)
correctness_df.to_csv(RESULTS/"operator_correctness.csv",index=False)
correctness_df

## 5. Workload reuse measurements

In [ ]:
SYSTEM="You are a repository analysis agent. Preserve exact symbol names, cite file paths, and never invent test results."
TOOLS=json.dumps({"tools":[
    {"name":"search_repository","parameters":{"type":"object","properties":{"query":{"type":"string"},"path":{"type":"string"}},"required":["query"]}},
    {"name":"run_tests","parameters":{"type":"object","properties":{"target":{"type":"string"}},"required":["target"]}}
]},indent=2)
MEMORY="Python 3.11; preserve public APIs; migrations require rollback; tests use pytest and mypy."

def repo_text(limit=120000):
    parts=[]; total=0
    for root in [Path(torch.__file__).resolve().parent,Path(pd.__file__).resolve().parent]:
        for p in root.rglob("*.py"):
            try: t=p.read_text(encoding="utf-8",errors="ignore")
            except: continue
            z=f"\n# FILE: {p.relative_to(root)}\n{t}\n"; parts.append(z); total+=len(z)
            if total>=limit: return "".join(parts)[:limit]
    return "".join(parts)[:limit]

REPO=repo_text()
RAG="\n\n".join([x for x in valid_texts if len(x.strip())>300][:12])
queries=["Find shape validation.","Explain retries.","Find a memory leak.","List filesystem functions.",
         "Propose error tests.","Explain configuration precedence.","Locate serialization.","Explain caching."]
workload=[]
for i,q in enumerate(queries):
    workload += [
        {"workload":"system_prompt","request_id":f"s{i}","text":SYSTEM+"\nUser: "+q},
        {"workload":"tool_schema","request_id":f"t{i}","text":SYSTEM+"\n"+TOOLS+"\nUser: "+q},
        {"workload":"repository","request_id":f"r{i}","text":SYSTEM+"\n"+REPO+"\nUser: "+q},
        {"workload":"agent_memory","request_id":f"m{i}","text":SYSTEM+"\n"+MEMORY+"\nUser: "+q},
        {"workload":"rag","request_id":f"g{i}","text":SYSTEM+"\n"+RAG+"\nUser: "+q},
    ]

if USER_TRACE_JSONL and Path(USER_TRACE_JSONL).exists():
    workload=[json.loads(x) for x in Path(USER_TRACE_JSONL).read_text().splitlines() if x.strip()]

def chunk_stats(ids,size):
    chunks=[tuple(ids[i:i+size]) for i in range(0,len(ids)-size+1,size)]
    c=Counter(chunks); hits=sum(max(0,n-1) for n in c.values())
    return len(chunks),len(c),hits/max(1,len(chunks)),hits*size/max(1,len(ids))

def common_prefix(sequences):
    n=min(map(len,sequences))
    for i in range(n):
        if any(x[i]!=sequences[0][i] for x in sequences[1:]): return i
    return n

## 6. Shared-prefix benchmark

In [ ]:
MODELS={"GPT-2 Small":"gpt2","SmolLM2-135M":"HuggingFaceTB/SmolLM2-135M",
        "TinyLlama-1.1B":"TinyLlama/TinyLlama-1.1B-Chat-v1.0"}

def timed(fn,repeats):
    for _ in range(3): fn()
    sync(); z=[]
    for _ in range(repeats):
        s=time.perf_counter(); fn(); sync(); z.append((time.perf_counter()-s)*1000)
    return float(np.median(z))

def max_context(model):
    for k in ["max_position_embeddings","n_positions","max_sequence_length","seq_length"]:
        v=getattr(model.config,k,None)
        if v: return int(v)
    return 2048

@torch.no_grad()
def run_model(label,model_id):
    tok=AutoTokenizer.from_pretrained(model_id,use_fast=True)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    kw=dict(torch_dtype=AMP_DTYPE if torch.cuda.is_available() else torch.float32,low_cpu_mem_usage=True)
    try: model=AutoModelForCausalLM.from_pretrained(model_id,attn_implementation="sdpa",**kw)
    except: model=AutoModelForCausalLM.from_pretrained(model_id,**kw)
    model.to(DEVICE).eval()

    encoded=[]; reuse=[]
    for item in workload:
        ids=tok.encode(item["text"],add_special_tokens=False); encoded.append({**item,"ids":ids})
        for size in [8,16,32,64,128]:
            chunks,unique,hit,frac=chunk_stats(ids,size)
            reuse.append({"tokenizer":label,"workload":item["workload"],"request_id":item["request_id"],
                          "tokens":len(ids),"chunk_size":size,"chunks":chunks,"unique_chunks":unique,
                          "chunk_hit_rate":hit,"repeatable_token_fraction":frac})
    prefixes=[]
    groups=defaultdict(list)
    for x in encoded: groups[x["workload"]].append(x["ids"])
    for name,seqs in groups.items():
        p=common_prefix(seqs); total=sum(map(len,seqs))
        prefixes.append({"tokenizer":label,"workload":name,"requests":len(seqs),"shared_prefix_tokens":p,
                         "total_request_tokens":total,"cross_request_reusable_fraction":p*(len(seqs)-1)/max(1,total)})

    base=SYSTEM+"\n"+TOOLS+"\n"+MEMORY+"\n"+REPO
    all_prefix=tok(base,return_tensors="pt",add_special_tokens=False).input_ids[0]
    suffix=tok("\nUser: Explain caching.\nAssistant:",return_tensors="pt",add_special_tokens=False).input_ids[0][:16]
    rows=[]
    for requested in CFG["prefill_contexts"]:
        n=min(requested,max_context(model)-32)
        if n<64: continue
        prefix=all_prefix[:n-16]
        full=torch.cat([prefix,suffix]).unsqueeze(0).to(DEVICE)
        pbatch=prefix.unsqueeze(0).to(DEVICE); sbatch=suffix.unsqueeze(0).to(DEVICE)
        full_fn=lambda: model(full,use_cache=False).logits[:,-1]
        first=model(pbatch,use_cache=True); cache=first.past_key_values
        if hasattr(cache,"to_legacy_cache"): cache=cache.to_legacy_cache()
        cached_fn=lambda: model(sbatch,past_key_values=cache,use_cache=True).logits[:,-1]
        a=timed(full_fn,CFG["prefill_repeats"]); b=timed(cached_fn,CFG["prefill_repeats"])
        err=float((full_fn().float()-cached_fn().float()).abs().max())
        rows.append({"model":label,"model_id":model_id,"parameters":params(model),"context_tokens":int(full.shape[1]),
                     "prefix_tokens":int(pbatch.shape[1]),"suffix_tokens":int(sbatch.shape[1]),
                     "full_prefill_ms":a,"cached_suffix_ms":b,"speedup":a/b,"max_logit_error":err})
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pd.DataFrame(rows),pd.DataFrame(reuse),pd.DataFrame(prefixes)

benches=[]; reuses=[]; prefixes=[]; failures=[]
if RUN_PRETRAINED_BENCHMARKS:
    for label,mid in MODELS.items():
        try:
            a,b,c=run_model(label,mid); benches.append(a); reuses.append(b); prefixes.append(c)
        except Exception as e:
            failures.append({"model":label,"model_id":mid,"error":repr(e)}); print(label,e)

prefill_df=pd.concat(benches,ignore_index=True) if benches else pd.DataFrame()
reuse_df=pd.concat(reuses,ignore_index=True) if reuses else pd.DataFrame()
prefix_df=pd.concat(prefixes,ignore_index=True) if prefixes else pd.DataFrame()
pd.DataFrame(failures).to_csv(RESULTS/"pretrained_failures.csv",index=False)
prefill_df.to_csv(RESULTS/"pretrained_prefill.csv",index=False)
reuse_df.to_csv(RESULTS/"workload_chunk_reuse.csv",index=False)
prefix_df.to_csv(RESULTS/"workload_prefix_reuse.csv",index=False)
prefill_df

## 7. Triton repeated-segment benchmark

In [ ]:
try:
    import triton
    import triton.language as tl
    TRITON_OK=torch.cuda.is_available()
except Exception as e:
    TRITON_OK=False; print(e)

if TRITON_OK:
    @triton.jit
    def token_kernel(a,b,h,out,L,R,D,BLOCK:tl.constexpr):
        offs=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK); mask=offs<D
        x=tl.load(h+offs,mask=mask,other=0.).to(tl.float32)
        for t in tl.range(0,L*R):
            row=(t%L)*D+offs
            x=tl.load(a+row,mask=mask,other=1.)*x+tl.load(b+row,mask=mask,other=0.)
        tl.store(out+offs,x,mask=mask)

    @triton.jit
    def power_kernel(sa,sb,h,out,R,D,BLOCK:tl.constexpr):
        offs=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK); mask=offs<D
        ba=tl.load(sa+offs,mask=mask,other=1.).to(tl.float32)
        bb=tl.load(sb+offs,mask=mask,other=0.).to(tl.float32)
        ra=tl.full((BLOCK,),1.,tl.float32); rb=tl.full((BLOCK,),0.,tl.float32); n=R
        for _ in tl.static_range(0,32):
            bit=n&1; ca=ba*ra; cb=ba*rb+bb
            ra=tl.where(bit!=0,ca,ra); rb=tl.where(bit!=0,cb,rb)
            bb=ba*bb+bb; ba=ba*ba; n=n>>1
        x=tl.load(h+offs,mask=mask,other=0.).to(tl.float32)
        tl.store(out+offs,ra*x+rb,mask=mask)

triton_rows=[]
if RUN_TRITON_BENCHMARKS and TRITON_OK:
    D=1024; BLOCK=256; grid=(triton.cdiv(D,BLOCK),)
    for L in [16,32,64,128]:
        a=.995+.004*torch.rand(L,D,device="cuda"); b=.02*torch.randn(L,D,device="cuda")
        h=torch.randn(D,device="cuda"); out=torch.empty_like(h); sa,sb=compile_op(a,b)
        for R in [1,4,16,64,256,1024]:
            f=lambda: token_kernel[grid](a,b,h,out,L,R,D,BLOCK=BLOCK)
            g=lambda: power_kernel[grid](sa,sb,h,out,R,D,BLOCK=BLOCK)
            f(); g(); sync()
            ta=float(triton.testing.do_bench(f)); tb=float(triton.testing.do_bench(g))
            f(); x=out.clone(); g(); y=out.clone()
            triton_rows.append({"segment_length":L,"repetitions":R,"context_tokens":L*R,
                                "tokenwise_ms":ta,"powered_ms":tb,"speedup":ta/tb,
                                "max_abs_error":float((x-y).abs().max())})
triton_df=pd.DataFrame(triton_rows)
triton_df.to_csv(RESULTS/"triton_operator.csv",index=False)
triton_df

## 8. Figures, tables, and archive

In [ ]:
if not prefill_df.empty:
    for name,g in prefill_df.groupby("model"):
        plt.plot(g["context_tokens"],g["speedup"],marker="o",label=name)
    plt.xlabel("Context tokens"); plt.ylabel("Full prefill / cached suffix"); plt.legend()
    plt.tight_layout(); plt.savefig(FIGURES/"pretrained_prefill.png",dpi=180); plt.show()

if not prefix_df.empty:
    prefix_df.pivot(index="workload",columns="tokenizer",values="cross_request_reusable_fraction").plot(kind="bar")
    plt.ylabel("Reusable token fraction"); plt.tight_layout()
    plt.savefig(FIGURES/"workload_reuse.png",dpi=180); plt.show()

if not triton_df.empty:
    for L,g in triton_df.groupby("segment_length"):
        plt.plot(g["context_tokens"],g["speedup"],marker="o",label=f"L={L}")
    plt.xscale("log",base=2); plt.yscale("log",base=2); plt.legend()
    plt.xlabel("Context tokens"); plt.ylabel("Powered speedup"); plt.tight_layout()
    plt.savefig(FIGURES/"triton_speedup.png",dpi=180); plt.show()

def latex(df,name):
    (RESULTS/name).write_text(
        "% No results\n" if df.empty else df.to_latex(index=False,escape=True,float_format=lambda x:f"{x:.4f}"),
        encoding="utf-8"
    )

latex(quality_df,"quality.tex"); latex(paired_df,"paired.tex")
latex(prefill_df,"prefill.tex"); latex(prefix_df,"reuse.tex")
latex(correctness_df,"correctness.tex"); latex(triton_df,"triton.tex")

manifest["completed_utc"]=pd.Timestamp.utcnow().isoformat()
(RESULTS/"manifest_final.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

zip_path=Path("/kaggle/working/mosaic_kaggle_results.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(ROOT))
print("DOWNLOAD:",zip_path)

## Outputs

Save the executed notebook and `/kaggle/working/mosaic_kaggle_results.zip`.